In [ ]:
# -----------------------------
# Imports
# -----------------------------
import json
import heapq
import math

# -----------------------------
# Load data
# -----------------------------
G_file = "data/G.json"
Coord_file = "data/Coord.json"
Dist_file = "data/Dist.json"
Cost_file = "data/Cost.json"

with open(G_file, "r") as f:
    G = json.load(f)

with open(Coord_file, "r") as f:
    Coord = json.load(f)

with open(Dist_file, "r") as f:
    Dist = json.load(f)

with open(Cost_file, "r") as f:
    Cost = json.load(f)

# -----------------------------
# Normalize dictionary to [0,1]
# -----------------------------
def normalize_dict(d):
    values = list(d.values())
    min_v, max_v = min(values), max(values)
    if max_v - min_v == 0:
        return {k: 0 for k in d}  # avoid division by zero
    return {k: (v - min_v)/(max_v - min_v) for k, v in d.items()}

normalized_dist = normalize_dict(Dist)

# -----------------------------
# Precompute Euclidean heuristic
# -----------------------------
goal_node = "50"
x_goal, y_goal = Coord[goal_node]

heuristic = {}
for node, (x, y) in Coord.items():
    dx = x - x_goal
    dy = y - y_goal
    heuristic[node] = math.sqrt(dx**2 + dy**2)

normalized_heuristic = normalize_dict(heuristic)

# -----------------------------
# A* search with weighted f(n)
# -----------------------------
def a_star(G, Cost, normalized_dist, normalized_heuristic, start, goal, max_energy, alpha=1, beta=1):
    """
    Weighted A* search:
        f(n) = alpha*normalized_dist + beta*normalized_heuristic
        energy cost tracked separately to respect max_energy
    """
    pq = [(0, start, [start], 0)] 
    visited = dict() 
    nodes_expanded = 0

    while pq:
        f_n, node, path, energy_used = heapq.heappop(pq)
        nodes_expanded += 1

        # Skip if energy exceeded
        if energy_used > max_energy:
            continue

        # Check if we reached the goal
        if node == goal:
            return path, energy_used, nodes_expanded

        # Skip if already visited with lower energy
        if node in visited and visited[node] <= energy_used:
            continue
        visited[node] = energy_used

        # Explore neighbors
        for neighbor in G.get(node, []):
            edge_key = f"{node},{neighbor}"
            edge_cost = Cost.get(edge_key, float('inf'))
            dist = normalized_dist.get(edge_key, 0)
            h = normalized_heuristic.get(neighbor, 0)

            new_energy = energy_used + edge_cost
            f_new = alpha * dist + beta * h  # weighted priority

            heapq.heappush(pq, (f_new, neighbor, path + [neighbor], new_energy))

    return None, None, nodes_expanded

# -----------------------------
# Run search
# -----------------------------
start_node = "1"
max_energy = 287932
alpha = 1     # weight for normalized distance
beta = 2      # weight for normalized heuristic

path, energy_used, nodes_expanded = a_star(
    G, Cost, normalized_dist, normalized_heuristic, start_node, goal_node, max_energy, alpha, beta
)

if path:
    print(f"Path: {path}")
    print(f"Energy used: {energy_used}")
    print(f"Nodes expanded: {nodes_expanded}")
    print(f"Number of nodes in path: {len(path)}")
else:
    print("No path found within energy limit")

Path: ['1', '1363', '1358', '1357', '1359', '1280', '1287', '1371', '1373', '1374', '1375', '1300', '1294', '1293', '1291', '1290', '1285', '1284', '1283', '1282', '1255', '1253', '1260', '1259', '1249', '1246', '963', '964', '962', '1002', '952', '1000', '998', '994', '995', '996', '997', '1011', '1006', '1010', '1024', '1309', '1310', '1315', '1316', '1321', '1326', '1327', '1323', '2935', '2936', '2941', '2933', '2934', '2928', '2921', '2516', '2519', '2520', '2521', '2522', '2545', '2546', '2548', '2538', '2551', '2541', '2552', '2543', '2631', '2630', '2628', '2627', '2625', '2585', '2583', '2580', '2561', '2557', '2448', '2388', '2386', '2380', '2445', '2444', '2405', '2406', '2398', '2395', '2397', '2142', '2141', '2125', '2126', '2082', '2080', '2071', '1979', '1975', '1967', '1966', '1974', '1973', '1971', '1970', '1948', '1937', '1939', '1935', '1931', '1934', '1673', '1675', '1674', '1837', '1671', '1828', '1825', '1817', '1815', '1634', '1814', '1813', '1632', '1631', '1742